# 2_6 — HMM na janela N=64

Extensão do artigo DATA 2026 na janela N=64. Este notebook **não**
altera nenhum notebook que produziu número publicado: o registro do congresso continua sendo
`2_2_Modelo_HHM`, `2_3_Modelo_AR_HMM` e `2_4_Modelo_Gaussian_HMM`.

## Divisão de papéis

| Metade | Reporta |
|---|---|
| **HMM** (aqui) | estados de Viterbi, matrizes de transição, PMFs/parâmetros de emissão, taxas de sincronização, curvas AIC/BIC |
| **Identificação de sistemas** | FIT%, Theil U₂, Diebold-Mariano, diagnóstico de resíduos |

Portanto **não** se calcula NRMSE, acurácia de previsão nem U₂ aqui. Previsão é da outra metade.

## Janela e dado

N = 64 trimestres (2010-Q1 a 2025-Q4); o congresso usou 59 níveis / K=58 log-retornos.
As séries vêm de `results/data/`, geradas por `scripts/export_series_tratadas.py`,
e carregam o mesmo tratamento de outlier/missing do painel da dissertação — sem o qual a ALUP4
voltaria a ter `0.0` em 2010-06-30 e 2011-06-30 (dado faltante corrigido no Round 28).

## Protocolo de transformação

Quem escolhe a transformação é o teste de raiz unitária, não o gosto do analista
(Invariante 2 do painel: `d` vem do consenso ADF/KPSS/PP):

| Condição | Transformação |
|---|---|
| `d = 0` | nível, sem transformação |
| `d = 1` e série estritamente positiva | log-retorno `ln(x_t) − ln(x_{t−1})` (Eq. 1 do congresso) |
| `d = 1` e série cruza o zero | primeira diferença `x_t − x_{t−1}` |

O congresso aplicou log-retorno no ramo contínuo — reportado só para *Gross Income*, a única
linha estritamente positiva nas 4 empresas — e `pct_change` no ramo categórico, sem o qual
*Operational Expenses*, que cruza o zero, não teria taxa de sincronização.

Para o ramo **categórico** a distinção entre log-retorno e `pct_change` é imaterial em série
positiva: `ln(1+p)` é monótona crescente em `p` e o binning por quantis é invariante a
transformação monótona — verificado na etapa de validação, que devolve a mesma PMF sob as duas.


In [1]:
import sys, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
logging.getLogger('hmmlearn').setLevel(logging.ERROR)
sys.path.insert(0, str(Path.cwd()))

import hmm_n64 as H
from hmm_n64_robustness import stage_robustness
from hmm_n64_validation import stage_validation

DATA_DIR, OUT_DIR = H.DEFAULT_DATA_DIR, H.DEFAULT_OUT_DIR
combos, manifest = H.load_combos(DATA_DIR)
print(f'{len(combos)} combinações | N={manifest["n_observacoes"]} | '
      f'{manifest["periodo"][0]} .. {manifest["periodo"][1]}')


12 combinações | N=64 | 2010-03-31 .. 2025-12-31


## 1. Protocolo aplicado às 12 combinações

Consenso de raiz unitária e transformação resultante. `K` é o comprimento da série modelada:
64 quando não há diferenciação, 63 quando há.


In [2]:
pd.DataFrame([
    {'empresa': c.empresa, 'indicador': c.indicador, 'd': c.d, 'transform': c.transform,
     'K': len(c.x), 'ADF_p': round(c.testes['ADF']['p_value'], 4),
     'KPSS_p': round(c.testes['KPSS']['p_value'], 4),
     'PP_p': None if c.testes['PP']['p_value'] is None else round(c.testes['PP']['p_value'], 4)}
    for c in combos
])


,empresa,indicador,d,transform,K,ADF_p,KPSS_p,PP_p
0,ELET6,resultado_bruto,0,level,64,0.0000,0.1000,0.0000
1,ISAE4,resultado_bruto,1,log_return,63,0.9979,0.0100,0.9825
2,TAEE4,resultado_bruto,1,log_return,63,0.9217,0.0100,0.2213
3,ALUP4,resultado_bruto,1,log_return,63,0.5998,0.0100,0.2291
4,ELET6,lucro_consolidado,0,level,64,0.2263,0.1000,0.0000
5,ISAE4,lucro_consolidado,1,first_diff,63,0.7534,0.0100,0.0056
6,TAEE4,lucro_consolidado,1,log_return,63,0.1008,0.0100,0.0005
7,ALUP4,lucro_consolidado,1,log_return,63,0.1032,0.0100,0.0001
8,ELET6,despesas_operacionais,1,first_diff,63,0.4035,0.0100,0.0000
9,ISAE4,despesas_operacionais,1,first_diff,63,0.7708,0.0100,0.0000


## 2. Pacote base

Estados de Viterbi, matrizes de transição, emissões, curvas AIC/BIC sobre N ∈ {2,3,4,5} e
taxas de sincronização, para Categorical, GHMM, GMM-HMM e AR-HMM.

Os rótulos de estado são reordenados canonicamente pela média de emissão crescente. Sem isso,
comparar estados entre empresas compara rótulos arbitrários do Baum-Welch (*label switching*) —
o código do congresso comparava rótulos brutos.

⚠️ Custo: cerca de um minuto por combinação.


In [3]:
stage_base_ja_rodou = (OUT_DIR / 'protocolo.csv').exists()
if not stage_base_ja_rodou:
    H.stage_base(combos, OUT_DIR)
else:
    print('artefatos já em disco — apague OUT_DIR para recomputar')


artefatos já em disco — apague OUT_DIR para recomputar


In [4]:
aic_bic = pd.read_csv(OUT_DIR / 'aic_bic.csv')
melhor = (aic_bic.loc[aic_bic.groupby(['empresa','indicador','variante'])['AIC'].idxmin()]
          .pivot(index=['empresa','indicador'], columns='variante', values='n_estados'))
print('Número de estados que minimiza o AIC:')
melhor


Número de estados que minimiza o AIC:


variante                       arhmm  categorical  gaussian  gmm
empresa indicador                                               
ALUP4   despesas_operacionais      3            2         3    2
        lucro_consolidado          2            2         2    3
        resultado_bruto            3            2         2    2
ELET6   despesas_operacionais      3            2         2    2
        lucro_consolidado          2            2         2    3
        resultado_bruto            3            2         3    2
ISAE4   despesas_operacionais      2            2         4    2
        lucro_consolidado          3            2         4    3
        resultado_bruto            3            3         3    3
TAEE4   despesas_operacionais      3            3         3    2
        lucro_consolidado          2            2         3    2
        resultado_bruto            3            2         4    3

In [5]:
print('Sincronização cross-company (P=4):')
display(pd.read_csv(OUT_DIR / 'sincronizacao_cross_company.csv'))
print('Sincronização intra-company (P=3):')
display(pd.read_csv(OUT_DIR / 'sincronizacao_intra_company.csv'))


Sincronização cross-company (P=4):


,variante,indicador,indicator_label,P,K,coincidencias,taxa,taxa_pct
0,arhmm,resultado_bruto,Gross Income,4,62,4,0.064516,6.452
1,arhmm,lucro_consolidado,Gross Profit,4,62,3,0.048387,4.839
2,arhmm,despesas_operacionais,Operational Expenses,4,62,1,0.016129,1.613
3,categorical,resultado_bruto,Gross Income,4,63,0,0.000000,0.000
4,categorical,lucro_consolidado,Gross Profit,4,63,3,0.047619,4.762
5,categorical,despesas_operacionais,Operational Expenses,4,63,1,0.015873,1.587
6,gaussian,resultado_bruto,Gross Income,4,63,4,0.063492,6.349
7,gaussian,lucro_consolidado,Gross Profit,4,63,2,0.031746,3.175
8,gaussian,despesas_operacionais,Operational Expenses,4,63,3,0.047619,4.762
9,gmm,resultado_bruto,Gross Income,4,63,2,0.031746,3.175


Sincronização intra-company (P=3):


,variante,empresa,P,K,coincidencias,taxa,taxa_pct
0,arhmm,ALUP4,3,62,5,0.080645,8.065
1,arhmm,ELET6,3,62,10,0.161290,16.129
2,arhmm,ISAE4,3,62,15,0.241935,24.194
3,arhmm,TAEE4,3,62,10,0.161290,16.129
4,categorical,ALUP4,3,63,2,0.031746,3.175
5,categorical,ELET6,3,63,5,0.079365,7.937
6,categorical,ISAE4,3,63,13,0.206349,20.635
7,categorical,TAEE4,3,63,16,0.253968,25.397
8,gaussian,ALUP4,3,63,1,0.015873,1.587
9,gaussian,ELET6,3,63,8,0.126984,12.698


## 3. Robustez — janela de treino e semente do EM

Análises de robustez que não couberam no artigo do
congresso (limite de 8 páginas). Aqui é medido por convergência e estabilidade da
partição (Rand ajustado), não por NRMSE — previsão pertence à outra metade.


In [6]:
if not (OUT_DIR / 'robustez_janela.csv').exists():
    stage_robustness(combos, OUT_DIR)
pd.read_csv(OUT_DIR / 'robustez_janela.csv').groupby(['variante','janela'])['convergiu'].mean().unstack()


janela,25,30,35,40
variante,,,,
arhmm_N2,0.833333,0.916667,1.000000,1.000000
arhmm_N3,0.833333,0.833333,0.916667,0.833333
categorical,1.000000,1.000000,1.000000,1.000000
gaussian,1.000000,1.000000,1.000000,1.000000
gmm,1.000000,1.000000,1.000000,1.000000


## 4. Validação contra as tabelas publicadas

Recalcula a janela do congresso (até 2024-Q3) com a **receita publicada** — `pct_change`,
M=3 bins por quantis, `CategoricalHMM(n_components=3, n_iter=100, random_state=1)`, ajuste
único, comparação por rótulo bruto — sobre três versões do dado. Responde a duas perguntas:
qual versão do dado o congresso usou, e a qual código CVM cada nome de empresa corresponde.


In [7]:
stage_validation(DATA_DIR, OUT_DIR)


     variante_dado  n_obs_nivel             indicador      indicator_label  K  coincidencias  taxa_pct  publicado_pct  bate
             bruta           59       resultado_bruto         Gross Income 56              9    16.071          6.900 False
             bruta           59     lucro_consolidado         Gross Profit 56              4     7.143         10.345 False
             bruta           59 despesas_operacionais Operational Expenses 56              8    14.286         13.793 False
bruta_alup4_interp           59       resultado_bruto         Gross Income 58              4     6.897          6.900  True
bruta_alup4_interp           59     lucro_consolidado         Gross Profit 58              6    10.345         10.345  True
bruta_alup4_interp           59 despesas_operacionais Operational Expenses 58              8    13.793         13.793  True
           tratada           59       resultado_bruto         Gross Income 58              1     1.724          6.900 False
        